# 4. Fidelity

> **Runtime: GPU.** Runtime → Change runtime type → **GPU** recommended (distance metrics, discriminator); **CPU** also works, only slower.

The 83 fidelity metrics F1–F12 ([METRICS.md](../docs/METRICS.md)) of every series against val, train and OOT (and the length-matched `@len` variants for the comparison with `real`, RQ0).

In [ ]:
# ── Drive first: the project (and colab_setup.py) lives there ──
from google.colab import drive
drive.mount("/content/drive")

# ── setup: Drive, code, dependencies, paths ──
REPO = "/content/drive/MyDrive/Colab Notebooks/MarathonTS"   # project folder on Drive (or a git clone)
PROFILE = "min"            # "min": the Colab run (fast); "mid": min + the VAE per seed; "max": everything on
# results go to {REPO}/artifacts/<PROFILE>; set ARTIFACTS = "..." to change
exec(open(f"{REPO}/colab/colab_setup.py").read())

In [ ]:
# ── runtime check: this notebook expects a GPU ──
import torch
if not torch.cuda.is_available():
    print("WARNING: no GPU in this runtime — Runtime → Change runtime type → GPU (results would be slower and timings not comparable)")
else:
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── scope of this run: change freely; finished units are skipped on re-run ──
import dataclasses
cfg = dataclasses.replace(
    cfg,
    datasets=list(cfg.datasets),  # the profile's datasets (marathonts/bench/config.py DATASETS); or a list, e.g. ["ett", "my_ds"]
    methods=None,                # None → the suites on every pool + long_methods; or a list to run a subset
    baselines=None,              # None → long_methods (case study: CS9, CS10)
    # custom components (EXTENDING.md): plugin files + one line each, no code changes:
    # plugins=["my_plugins.py"],                          # files registering them (path from the project folder)
    # pool_suites={"vae": "tsgm_vae", "gan": "my_gan"},   # the whole suite on another pool generator → gan_zseries, …
    # extra_assemblers=["my_stitch"],                     # a stitcher on every pool → vae_my_stitch, …
    # extra_joiners=["my_smooth"],                        # your own seam processing (run like the built-in ones)
    # smooth_paths=["zseries", "greedy"],                 # stitchers that get seam processing ("all": all but the random path and vae_quilting; min: ["zseries"])
    # joiners=["crossfade", "wsola"],                     # which seam processings (None: all)
    # disabled=["pctsp", "chronos"],                      # switch off methods or components by name
    # noise_twin=True,                                    # + the noise twin vae_rand_b (min: off; mid, max: on)
    # frozen_methods=[],                                  # published results kept as they are, not run (default: vae_zseries and its
    #                                                     #   seam-processing variants, marathonts/bench/config.py FROZEN_METHODS); [] → left out of the tables
    # long_methods=[*cfg.long_methods, "my_long"],          # + a long-series generator (method and baseline)
    seeds=list(range(10)),
)
# plugins=[...] above loads custom components (datasets, generators, assemblers, joiners, models, metrics, tasks) in
# every notebook, in the stage subprocesses and in the hypothesis part — EXTENDING.md

In [ ]:
from marathonts.bench.run import stage_fidelity
stage_fidelity(cfg)